# (0) Title Section
**Title:** Predicting Hotel Cancellations Using Lead Time and Booking Characteristics  
**Name:** Bryan Liao  
**Group Number and Name:** [G17 / I love STAT301 ]  
**Group Members:** [Karen Li, Niall Calvert, Mengyi Zhang]

In [ ]:
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(broom))

# Hotel Booking Analysis
## (1) Data Description
This dataset consists of a random subset of 1,000 hotel bookings originally collected from a City Hotel and a Resort Hotel in Portugal. The data frame contains 32 variables (plus row names) covering booking details, stay durations, and guest demographics.

In [ ]:
# Load the dataset
data_url <- "https://vincentarelbundock.github.io/Rdatasets/csv/bayesrules/hotel_bookings.csv"
hotel_data <- read_csv(data_url)

# a) The number of observations and variables
dim(hotel_data)

# b) Table with the names and types of the variables
glimpse(hotel_data)

**c) Data Collection:** The original data was extracted directly from the Property Management Systems (PMS) of a resort and a city hotel in Portugal. This specific dataset represents a random sample of 1,000 bookings from those original records.

**d) Source & Citation:** Antonio, N., de Almeida, A. and Nunes, L. (2019) “Hotel booking demand datasets,” Data in Brief, 22, pp. 41–49.

## (2) Question
**a) Question:** Does a longer lead time significantly increase the probability of cancellation, and how does this effect differ between Resort and City hotels?

**b) Focus:** This question addresses both **inference** and **prediction**. I anticipate using **Logistic Regression (GLM)** to model the binary outcome.

**c) Response Variable:** `is_canceled` (1 = yes, 0 = no).

**d) Important Covariates:** `lead_time`, `hotel`, and `deposit_type`.

**e) Controls & Other Variables:** Most of the other variables will serve as covariates to control for confounding factors in the model. Specifically, demographic variables (e.g., `country`, `adults`, `children`) will control for guest profiles, guest history (e.g., `previous_cancellations`, `is_repeated_guest`) will account for reliability, and all remaining logistical details (e.g., `market_segment`, `average_daily_rate`, `special_requests`) will control for the financial and structural nature of the reservation.

## (3) Exploratory Data Analysis and Visualization

In [ ]:
# b) Clean and wrangle data
hotel_clean <- hotel_data %>%
    # Need to convert the string "NULL" to actual NA as glm function doesn't know it means "missing"
    mutate(across(where(is.character), ~na_if(., "NULL"))) %>%
    mutate(
        hotel = as.factor(hotel),
        is_canceled = as.factor(is_canceled),
        arrival_date_month = factor(arrival_date_month, levels = month.name),
        customer_type = as.factor(customer_type),
        deposit_type = as.factor(deposit_type),
        market_segment = as.factor(market_segment)
    )

# c) Missing values check for each variable
hotel_clean %>%
    summarise(across(everything(), ~sum(is.na(.)))) %>%
    pivot_longer(everything(), names_to = "Variable", values_to = "Missing_Count") %>%
    mutate(Prop_Missing = Missing_Count / nrow(hotel_clean))

# d) Class imbalance check for categorical variable
print("--- Class Imbalance: is_canceled ---")
hotel_clean %>%
    count(is_canceled) %>%
    mutate(Proportion = n / sum(n)) %>%
    print()

print("--- Class Imbalance: hotel ---")
hotel_clean %>%
    count(hotel) %>%
    mutate(Proportion = n / sum(n)) %>%
    print()

print("--- Class Imbalance: deposit_type ---")
hotel_clean %>%
    count(deposit_type) %>%
    mutate(Proportion = n / sum(n)) %>%
    print()

**Interpretation of Missing Values:** An analysis of missing values reveals that `country` has very few missing entries (0.2%) and `agent` has a little missing entries (12.8%). However, the `company` variable exceeds the 20% threshold with over 95% missing data. Per instructions, all variables, including agent and company, are preserved for this initial stage.

**Interpretation of Class Imbalance**: There is a moderate class imbalance in both the response variable `is_canceled` (63.4% not canceled vs. 36.6% canceled) and the `hotel` covariate (64.2% City Hotel vs. 35.8% Resort Hotel). However, there is severe class imbalance in the `deposit_type` variable, where 88.1% of bookings are "No Deposit" and only 0.3% are "Refundable"; this extreme skew must be carefully managed during modeling so it does not disproportionately bias the predictions.

In [ ]:
# e) High-quality visualization
options(repr.plot.width = 15, repr.plot.height = 10)

suppressWarnings({
    print(
        hotel_clean %>%
            ggplot(aes(x = customer_type, y = lead_time, fill = is_canceled)) +
            geom_violin(alpha = 0.4, color = NA, trim = FALSE, position = position_dodge(width = 0.8)) +
            geom_jitter(aes(color = is_canceled), 
                        position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.8), 
                        alpha = 0.5, size = 1.2) +
            geom_boxplot(width = 0.15, outlier.shape = NA, alpha = 0.7, 
                         position = position_dodge(width = 0.8), color = "black") +
            facet_wrap(~hotel) +
            coord_cartesian(ylim = c(0, 450)) +
            labs(
                title = "Lead Time Distribution by Cancellation Status and Customer Type",
                subtitle = "Faceted by Hotel Category. Jittered points reveal underlying sample sizes per group.",
                x = "Customer Category",
                y = "Lead Time (Days)",
                fill = "Canceled?",
                color = "Canceled?"
            ) +
            theme_minimal() +
            theme(
                text = element_text(size = 15),
                strip.text = element_text(face = "bold", size = 14),
                panel.grid.minor = element_blank()
            ) +
            scale_fill_manual(values = c("#2ca02c", "#d62728"), labels = c("No", "Yes")) +
            scale_color_manual(values = c("#2ca02c", "#d62728"), labels = c("No", "Yes"))
    )
})

**Visualization Relevance:** This faceted violin plot with overlaid jittered points is highly relevant because it visualizes the interaction between our response variable (`is_canceled`) and key covariates (`lead_time`, `customer_type`, and `hotel`). The addition of jittered points explicitly reveals that "Group" and "Contract" customer types have very small sample sizes, making their summary statistics less reliable than the heavily populated "Transient" group. Most importantly, the plot demonstrates that canceled bookings (red) consistently have a higher median lead time and a wider distribution spread, whereas non-canceled bookings (green) cluster heavily near a lead time of zero, suggesting that bookings made further in advance carry a significantly higher risk of cancellation across all customer groups.

## Assignment 2

## (1) Methods and Plan

**a) Reviewed Question**  
Which combination of booking characteristics, logistics, and guest demographics are the most significant predictors of a hotel cancellation? (Note: Since this is an observational study, our focus is on predictive association, not causation).

**b) Proposed Method**  
I will use a Data Splitting & Stepwise Logistic Regression approach. First, to avoid "double-dipping" (using the same data to pick variables and then test them), I will split the data into a 70% training set and a 30% testing set. Before doing any modeling, I will combine tiny categories (like 'Refundable' or 'Group') into larger groups. This prevents mathematical crashes caused by rare events getting lost when the data is split.

To pick the best variables, I will run Stepwise Selection on the training set. However, instead of the default AIC metric, I will use the Bayesian Information Criterion (BIC). BIC applies a much stricter penalty for adding variables. This forces the model to stay simple and helps filter out "fake" signals (noise) that Stepwise often accidentally picks up. Finally, I will run a standard Multiple Logistic Regression on the unseen 30% testing set using only the variables chosen by BIC. This gives us honest, unbiased Odds Ratios for our final results.

**c) Assumptions and Limitations**
* **Independence of Observations:** The model assumes each booking is independent. A limitation is that some bookings might be linked (e.g., a family booking multiple rooms or a travel agency making bulk bookings), which would violate this assumption.
* **Linearity in the Logit:** We assume a linear relationship between the continuous predictor (`previous_cancellations`) and the log-odds of the response. 
* **Multicollinearity:** We assume our predictors are not highly correlated with one another.
* **Causal Limitations:** Because this dataset was collected via an observational study (data scraped from PMS systems) rather than a randomized controlled trial, a major limitation is that we cannot make causal interpretations. We can only state that certain features are *associated* with higher risks of cancellation.

## (2) Computational Code and Output

**a) Computation Code**
*(Note: Ensure `hotel_clean` from Assignment 1 is already loaded in the environment).*

In [ ]:
# a) Computation Code
set.seed(2026) # Ensures reproducibility for the TA grading

# Prepare and Split the Data (70% Train, 30% Test)
hotel_model_data <- hotel_clean %>% 
    # collapse tiny groups to prevent 'complete separation' crashes
    mutate(
        deposit_type = fct_collapse(deposit_type, "Has Deposit" = c("Non Refund", "Refundable")),
        customer_type = fct_collapse(customer_type, "Other" = c("Group", "Contract")),
        market_segment = fct_collapse(market_segment, "Other" = c("Complementary"))
    ) %>%
    # Remove NA-heavy columns and 'target leakage' columns that cheat the prediction
    select(
        -arrival_date_month, 
        -country, 
        -company,                 
        -agent,
        # target leakage columns
        -reservation_status,      
        -reservation_status_date
    ) %>%
    # Safety net: drop any column that accidentally has only 1 unique value left
    select(where(~n_distinct(.) > 1))

# 70/30 split
train_indices <- sample(seq_len(nrow(hotel_model_data)), size = 0.7 * nrow(hotel_model_data))
train_data <- hotel_model_data[train_indices, ]
test_data <- hotel_model_data[-train_indices, ]

# Fit the Null and Full models on the TRAINING data
null_model <- glm(is_canceled ~ 1, data = train_data, family = "binomial")
full_model <- glm(is_canceled ~ ., data = train_data, family = "binomial")

# Perform Stepwise Selection
# trace = 0 to not print the massive selection algorithm steps
suppressWarnings({
    # Calculate the log of the number of rows in the training data for the BIC penalty
    bic_penalty <- log(nrow(train_data))
    
    # Run stepwise using k = bic_penalty
    step_model <- step(null_model, scope = formula(full_model), direction = "both", trace = 0, k = bic_penalty)
    
    # Fit the Final Inference Model on the TESTING data using the selected formula
    final_test_model <- glm(formula(step_model), data = test_data, family = "binomial")
    
    # Create table (Odds Ratios)
    model_results <- tidy(final_test_model, exponentiate = TRUE, conf.int = TRUE) %>%
        select(term, estimate, conf.low, conf.high, p.value) %>%
        mutate_if(is.numeric, round, 3)
})

# Display the clean results table
model_results

# table(hotel_clean$customer_type)
# table(hotel_clean$market_segment)
# table(hotel_clean$deposit_type)

**c) Interpretation of Results**   
The model showed that only `previous_cancellations`, `adults`, and `total_of_special_requests` were statistically significant predictors on our testing data. Having a history of past cancellations (OR = 6.735) and adding more adults to a booking (Odd Ratio = 2.366) strongly increase the risk of a cancellation, while making special requests lowers the risk (Odd Ratio = 0.278). Unexpectedly, variables like `deposit_type` and `market_segment` produced massive errors (like infinite Odds Ratios). This happened because of "complete separation", when we randomly split our data 70/30, some rare categories ended up with too few examples for the math to calculate properly. To fix this mathematical breakdown in our final project, we will need to use an advanced method called Group LASSO to handle these small, unstable categories safely.